### 0: Downloading Programmes

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
import pandas as pd
from pathlib import Path
from collections import defaultdict
import time as time_module


### 1: Dataframes

In [3]:
base = Path("output_folder")


df_resources = pd.read_csv(base / "resources.csv") # resource ID and resource type
df_constraints = pd.read_csv(base / "constraints.csv") # constraint ID and type, Name, required?, weight and cost function

df_times = pd.read_csv(base / "times.csv").sort_values("order_index") # timeslots and their timegroups (days)



# df_events = event with its duration, course, groups, role, classes and teacher)
events = pd.read_csv(base / "events.csv") 
event_groups = pd.read_csv(base / "event_eventgroup_membership.csv")
event_resources = pd.read_csv(base / "event_resources.csv")

# Events each group is part of
groups_per_event = (
    event_groups
    .groupby("event_id")["event_group_id"]
    .apply(list)
    .reset_index(name="groups")
)

# Teacher of each event
teachers_per_event = event_resources[event_resources["role"] == "Teacher"]
teachers_per_event = ( teachers_per_event[["event_id", "reference_resource_id"]]
    .groupby("event_id")["reference_resource_id"]
    .apply(list)
    .reset_index(name="teachers")
) 

#Classes in each event
classes_per_event = event_resources[event_resources["role"].astype(str).str.startswith("Class")]
classes_per_event = ( classes_per_event[["event_id", "reference_resource_id"]]
    .groupby("event_id")["reference_resource_id"]
    .apply(list)
    .reset_index(name="classes")
)

# Role of each event
role_per_event = event_resources[event_resources["resource_type_ref"] == "Room"]
role_per_event = (role_per_event[["event_id", "role"]]
    .groupby("event_id")["role"]
    .first()
    .reset_index(name="role")
)

df_events = (
    events.rename(columns={"course_ref": "course_reference"})[["event_id", "duration", "course_reference"]]
    .merge(groups_per_event, on="event_id", how="left")
    .merge(role_per_event, on="event_id", how="left")
    .merge(classes_per_event, on="event_id", how="left")
    .merge(teachers_per_event, on="event_id", how="left")
) # event with its duration, course, groups, role, classes and teacher)


resource_groups = pd.read_csv(base / "resource_group_membership.csv")

# Keep only what we need from memberships and aggregate group memberships
groups_per_resource = (
    resource_groups
    .groupby("resource_id")["resource_group_id"]
    .apply(list)
    .reset_index(name="resource_groups")
)

# Merge onto resources
df_res_with_groups = (
    df_resources[["resource_id", "resource_type_ref"]]
    .merge(groups_per_resource, on="resource_id", how="left")
)

# Replace NaN with empty list (resources that have no group membership)
df_res_with_groups["resource_groups"] = df_res_with_groups["resource_groups"].apply(
    lambda x: x if isinstance(x, list) else []
)

# Split into separate tables
df_teachers = df_res_with_groups[df_res_with_groups["resource_type_ref"] == "Teacher"].reset_index(drop=True) # teacher id, resource type and list of groups
df_classes  = df_res_with_groups[df_res_with_groups["resource_type_ref"] == "Class"].reset_index(drop=True) # class id, resource type and list of groups
df_rooms    = df_res_with_groups[df_res_with_groups["resource_type_ref"] == "Room"].reset_index(drop=True) # room id, resource type and list of groups


df_res_long = (
    df_resources[["resource_id", "resource_type_ref"]]
    .merge(resource_groups[["resource_id", "resource_group_id"]], on="resource_id", how="left")
    .rename(columns={"resource_group_id": "resource_group"})
)
df_teachers_long = df_res_long[df_res_long["resource_type_ref"] == "Teacher"] # teacher id, resource type and seperate groups
df_classes_long  = df_res_long[df_res_long["resource_type_ref"] == "Class"] # class id, resource type and seperate groups
df_rooms_long    = df_res_long[df_res_long["resource_type_ref"] == "Room"] # room id, resource type and seperate groups




df_constraint_applied = pd.read_csv(base / "constraint_applies_to.csv")
df_constraint_params = pd.read_csv(base / "constraint_params.csv")

df_constraint_params = df_constraint_params.merge(
    df_constraints[["constraint_id", "constraint_type"]],
    on="constraint_id",
    how="left"
)

In [5]:
#print(df_resources)
#print(df_constraints)

#print(df_times)
#print(df_events)
#print(df_rooms)
#print(df_teachers)
#print(df_classes)

#df_constraint_params



### 2a: Model setup and helpers

In [ ]:
# function to return things as list
def as_list(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    return [x]




### Sets
E = df_events["event_id"].tolist() # All events
T = df_times["time_id"].tolist() # All timeslots
R = df_rooms["resource_id"].tolist() # All rooms

all_teachers = df_teachers["resource_id"].tolist()
all_classes = df_classes["resource_id"].tolist()


# event -> duration/teachers/classes (lists)
event_to_duration = dict(zip(df_events["event_id"], df_events["duration"].astype(int))) # duration of each event
event_to_teachers = dict(zip(df_events["event_id"], df_events["teachers"])) # teachers for each event
event_to_classes  = dict(zip(df_events["event_id"], df_events["classes"])) # classes for each event


# event -> room group of each event
event_to_roomgroup = dict(zip(df_events["event_id"], df_events["role"])) 

# room -> room group of each room
room_groups = {"gr_Rooms_X", "gr_Rooms_Y", "gr_Rooms_Z"}
room_to_group = (
    df_rooms_long[df_rooms_long["resource_group"].isin(room_groups)]
    .set_index("resource_id")["resource_group"]
    .to_dict()
)

# room group -> rooms in that group
group_to_rooms = defaultdict(list)
for room, grp in room_to_group.items():   # room_to_group: room -> gr_Rooms_X/Y/Z
    group_to_rooms[grp].append(room)


T_index = {t:i for i,t in enumerate(T)}

df_unavail_times = df_constraint_params[
    (df_constraint_params["constraint_type"] == "AvoidUnavailableTimesConstraint") &
    (df_constraint_params["path"] == "Times/Time")
].copy()

df_unavail_times = df_unavail_times[["reference", "value", "constraint_id"]]

# df_unavail_times has: reference (teacher), value (times teacher not available), constraint_id
unavailability = (
    df_unavail_times
    .groupby("constraint_id")
    .agg(
        teacher=("reference", "first"),
        times=("value", lambda s: sorted(set(s), key=lambda t: T_index[t]))
    )
    .to_dict(orient="index")
)

In [ ]:
### Feasible Start times for events:

# time -> day 
time_to_day = dict(zip(df_times["time_id"], df_times["day_ref"]))

# Build feasible starts per event (no crossing day boundary) 
feasible_starts = {}
for e in E:
    d = int(event_to_duration[e]) # number of periods needed
    starts = []
    for ts in T: # for all possible start times
        i0 = T_index[ts]
        # must have enough periods remaining in the global list
        if i0 + d - 1 >= len(T):
            continue
        day0 = time_to_day[ts]
        block = T[i0:i0 + d]
        # enforce all times in the block are same day
        if all(time_to_day[t] == day0 for t in block):
            starts.append(ts)
    feasible_starts[e] = starts

# list of occupied times for each (event, starttime) pair
occ_times = {}
for e in E:
    d = int(event_to_duration[e])
    for ts in feasible_starts[e]:
        i0 = T_index[ts]
        occ_times[(e, ts)] = T[i0:i0 + d]


In [ ]:
feasible_starts_filtered = {}

for e in E:
    starts_ok = []
    for ts in feasible_starts[e]:
        occ = occ_times[(e, ts)]  # occupied periods if e starts at ts

        # Teacher-specific unavailability constraints
        violates = False
        for info in unavailability.values():
            teacher = info["teacher"]
            unavail_set = set(info["times"])
            if teacher in as_list(event_to_teachers[e]):
                # if any occupied time is unavailable -> illegal
                if any(t in unavail_set for t in occ):
                    violates = True
                    break

        if not violates:
            starts_ok.append(ts)

    feasible_starts_filtered[e] = starts_ok


# Replace
feasible_starts = feasible_starts_filtered



# list of occupied times for each (event, starttime) pair
occ_times = {}
for e in E:
    d = int(event_to_duration[e])
    for ts in feasible_starts[e]:
        i0 = T_index[ts]
        occ_times[(e, ts)] = T[i0:i0 + d]

# all (e, ts) pairs that occupy a given time t
# ie for Fr_3, if event 1 is duration 1 and event 2 is duration 2, then (1, Fr_3) and (2, Fr_2) and (2, Fr_3) occupy it
occupies_at_time = {t: [] for t in T}
for e in E:
    for ts in feasible_starts[e]:
        for t in occ_times[(e, ts)]:
            occupies_at_time[t].append((e, ts))

In [ ]:
#### For explicit Feasible Starts constraints

time_to_day = dict(zip(df_times["time_id"], df_times["day_ref"]))

all_starts = {e: list(T) for e in E}

# Build occupied-time blocks for every candidate start
occ_times = {}
for e in E:
    d = int(event_to_duration[e])
    for ts in T:
        i0 = T_index[ts]
        if i0 + d - 1 < len(T):
            occ_times[(e, ts)] = T[i0:i0 + d]
        else:
            occ_times[(e, ts)] = []


occupies_at_time = {t: [] for t in T}
for e in E:
    for ts in T:
        for t in occ_times[(e, ts)]:
            occupies_at_time[t].append((e, ts))

feasible_starts = all_starts



### 2b: Pyomo Model part 1: Sets and Decision Variables

Constraints:

Part 1: Event -> time (hard constraints):

4) AssignTimes: [gr_EventsGeneral] Every event must be assigned a time
5) Do Not Split Events: [gr_EventsGeneral] Events must inhabit consecutive periods
9) PreferredTimes: [gr_EventsDuration1] Events with duration 1 must only be allocated to times in groups gr_TimesDurationTwo
10) PreferredTimesDurationTwo: [gr_EventsDuration1] ...
11) PreferredTimesDurationThree: [gr_EventsDuration1] ...
12) PreferredTimesDurationFour: [gr_EventsDuration1] ...
13) EventSpreadingConstraint: [gr_001 - gr_150] events from each goup have a maximum and minimum specified number of events in each time group [monday, tuesday etcc...] (here max 1 per day) (event group given in course ID)
14) NoResourceClashes: No clashes for any of the resources
15) Unavailabilities_T07: [T07] selected Teacher cannot be allocated event at any of the given times, 
...
25) Unavailabilities_T17: ...

### 2b: Start-Time Based Model

In [ ]:
# ---- Model ----
m1 = pyo.ConcreteModel()

# ---- Sets ----
m1.E = pyo.Set(initialize=E) # Set of events
m1.T = pyo.Set(initialize=T, ordered=True) # Set of timeslots (ordered)
m1.R = pyo.Set(initialize=R) # Set of rooms

m1.Teachers = pyo.Set(initialize=all_teachers) # set of all teachers
m1.Classes = pyo.Set(initialize=all_classes) # set of all classes

# standard set up
m1.X_index = pyo.Set(
    dimen=2,
    initialize=[(e, ts) for e in E for ts in feasible_starts[e]]
)

m1.x = pyo.Var(m1.X_index, domain=pyo.Binary) # = 1 if event e starts at time ts

In [ ]:

# Standard set up

# Each event scheduled exactly once (one start time)  
def one_start_rule(m, e):
    return sum(m.x[e, ts] for ts in feasible_starts[e]) == 1
m1.OneStart = pyo.Constraint(m1.E, rule=one_start_rule)



# No teacher conflicts: a teacher can teach at most one event at a time
def teacher_conflict_rule(m, teacher, t):
    terms = [
        m.x[e, ts]
        for (e, ts) in occupies_at_time.get(t, [])
        if teacher in as_list(event_to_teachers[e])
    ]
    if not terms:
        return pyo.Constraint.Skip   # or pyo.Constraint.Feasible
    return sum(terms) <= 1

m1.TeacherConflict = pyo.Constraint(all_teachers, m1.T, rule=teacher_conflict_rule)

# No class conflicts: a class can attend at most one event at a time
def class_conflict_rule(m, cls, t):
    terms = [
        m.x[e, ts]
        for (e, ts) in occupies_at_time.get(t, [])
        if cls in event_to_classes[e]
    ]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

m1.ClassConflict = pyo.Constraint(all_classes, m1.T, rule=class_conflict_rule)

In [ ]:
#### FOR EXPLICIT FEASIBLE START CONSTRAINTS

start_allowed = {}

for e in E:
    dur = int(event_to_duration[e])
    teachers_e = set(as_list(event_to_teachers[e]))

    for ts in T:
        i0 = T_index[ts]

        # enough periods left in horizon
        if i0 + dur - 1 >= len(T):
            start_allowed[(e, ts)] = 0
            continue

        block = T[i0:i0 + dur]
        day0 = time_to_day[ts]

        # must stay within one day
        if not all(time_to_day[t] == day0 for t in block):
            start_allowed[(e, ts)] = 0
            continue

        # teacher unavailability
        violates_unavailability = False
        for info in unavailability.values():
            teacher = info["teacher"]
            if teacher in teachers_e:
                unavail_set = set(info["times"])
                if any(t in unavail_set for t in block):
                    violates_unavailability = True
                    break

        start_allowed[(e, ts)] = 0 if violates_unavailability else 1


m1.start_allowed = pyo.Param(
    m1.X_index,
    initialize=start_allowed,
    within=pyo.Binary
)

def valid_start_rule(m, e, ts):
    return m.x[e, ts] <= m.start_allowed[e, ts]

m1.ValidStart = pyo.Constraint(m1.X_index, rule=valid_start_rule)




In [ ]:
### Events that are part of the same course must be on different days

# Map event -> course_reference
event_to_course = dict(zip(df_events["event_id"], df_events["course_reference"]))

# Build course groups (gr_C001 ... gr_C150)
courses = sorted(set(event_to_course[e] for e in E))

# Days (time groups) from df_times (gr_Mo, gr_Tu, ...)
days = sorted(df_times["day_ref"].dropna().unique().tolist())

# Collect relevant x vars by (course, day)
# key -> list of all feasible (e, ts) combinations for given course and day
course_day_terms = defaultdict(list)

for (e, ts) in m1.X_index:
    c = event_to_course[e]
    d = time_to_day[ts]   # day of the start time
    course_day_terms[(c, d)].append((e, ts))

# Define Pyomo sets for indexing
m1.COURSES = pyo.Set(initialize=courses)
m1.DAYS = pyo.Set(initialize=days)

# Max-per-day constraint (Maximum = 1)
def event_spread_max_rule(m, c, d):
    terms = course_day_terms.get((c, d), [])
    if not terms:
        return pyo.Constraint.Skip
    return sum(m.x[e, ts] for (e, ts) in terms) <= 1 # at most one event for every course each day

m1.EventSpreadMax = pyo.Constraint(m1.COURSES, m1.DAYS, rule=event_spread_max_rule)

In [37]:
# ---- Objective ---- (hard constraints only)
m1.obj = pyo.Objective(expr=0, sense=pyo.minimize)

In [ ]:
### Preprocessing Feasible Times
seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(m1)

    solver = pyo.SolverFactory("gurobi")
    solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    # Manual timing
    start_time = time_module.time()
    res = solver.solve(m1, tee=False)
    elapsed_time = time_module.time() - start_time

    solve_times.append(elapsed_time)
    objectives.append(pyo.value(m1.obj))
    num_vars.append(m1.nvariables())
    num_cons.append(m1.nconstraints())

    solve_status.append(res.solver.termination_condition)

    # Store solution
    seed_solution = {}
    for (e, ts) in m1.X_index:
        val = pyo.value(m1.x[e, ts], exception=False)
        seed_solution[(e, ts)] = 0.0 if val is None else float(val)
    start_x_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x = start_x_list[0]
res = res

# Calculate averages
avg_time = sum(solve_times) / len(solve_times)
avg_obj = sum(objectives) / len(objectives)
avg_vars = sum(num_vars) / len(num_vars)
avg_cons = sum(num_cons) / len(num_cons)

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {avg_obj:.6f}")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Termination Status: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    scheduled = sum(1 for v in seed_sol.values() if v > 0)
    print(f"  Seed {seeds[i]}: {scheduled} events scheduled in {solve_times[i]:.2f}s, Obj={objectives[i]:.6f}")


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 11.5222
Average Objective: 0.000000
Average Variables: 2650
Average Constraints: 1376
Termination Status: {<TerminationCondition.optimal: 'optimal'>}

Phase 1 produced 10 solutions (one per seed)
  Seed 10: 169 events scheduled in 7.92s, Obj=0.000000
  Seed 20: 169 events scheduled in 12.43s, Obj=0.000000
  Seed 30: 169 events scheduled in 15.16s, Obj=0.000000
  Seed 40: 169 events scheduled in 6.78s, Obj=0.000000
  Seed 50: 169 events scheduled in 6.66s, Obj=0.000000
  Seed 60: 169 events scheduled in 16.70s, Obj=0.000000
  Seed 70: 169 events scheduled in 5.40s, Obj=0.000000
  Seed 80: 169 events scheduled in 17.75s, Obj=0.000000
  Seed 90: 169 events scheduled in 18.06s, Obj=0.000000
  Seed 100: 169 events scheduled in 8.37s, Obj=0.000000


In [ ]:
### Explicit Feasible Times Constraints
seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(m1)

    solver = pyo.SolverFactory("gurobi")
    solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    # Manual timing
    start_time = time_module.time()
    res = solver.solve(m1, tee=False)
    elapsed_time = time_module.time() - start_time

    solve_times.append(elapsed_time)
    objectives.append(pyo.value(m1.obj))
    num_vars.append(m1.nvariables())
    num_cons.append(m1.nconstraints())

    solve_status.append(res.solver.termination_condition)

    # Store solution
    seed_solution = {}
    for (e, ts) in m1.X_index:
        val = pyo.value(m1.x[e, ts], exception=False)
        seed_solution[(e, ts)] = 0.0 if val is None else float(val)
    start_x_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x = start_x_list[0]
res = res

# Calculate averages
avg_time = sum(solve_times) / len(solve_times)
avg_obj = sum(objectives) / len(objectives)
avg_vars = sum(num_vars) / len(num_vars)
avg_cons = sum(num_cons) / len(num_cons)

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {avg_obj:.6f}")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Termination Status: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    scheduled = sum(1 for v in seed_sol.values() if v > 0)
    print(f"  Seed {seeds[i]}: {scheduled} events scheduled in {solve_times[i]:.2f}s, Obj={objectives[i]:.6f}")


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 7.4224
Average Objective: 0.000000
Average Variables: 3380
Average Constraints: 4999
Termination Status: {<TerminationCondition.optimal: 'optimal'>}

Phase 1 produced 10 solutions (one per seed)
  Seed 10: 169 events scheduled in 4.59s, Obj=0.000000
  Seed 20: 169 events scheduled in 6.83s, Obj=0.000000
  Seed 30: 169 events scheduled in 10.66s, Obj=0.000000
  Seed 40: 169 events scheduled in 4.51s, Obj=0.000000
  Seed 50: 169 events scheduled in 3.70s, Obj=0.000000
  Seed 60: 169 events scheduled in 10.37s, Obj=0.000000
  Seed 70: 169 events scheduled in 3.44s, Obj=0.000000
  Seed 80: 169 events scheduled in 12.04s, Obj=0.000000
  Seed 90: 169 events scheduled in 12.39s, Obj=0.000000
  Seed 100: 169 events scheduled in 5.71s, Obj=0.000000


### Occupancy - Start Time Hybrid

In [ ]:
# Build all candidate starts and occupancy coverage
covering_starts = {(e, t): [] for e in E for t in T}
occ_times = {}

for e in E:
    dur = int(event_to_duration[e])
    for ts in T:
        i0 = T_index[ts]
        if i0 + dur - 1 < len(T):
            block = T[i0:i0 + dur]
            occ_times[(e, ts)] = block
            for t in block:
                covering_starts[(e, t)].append(ts)
        else:
            occ_times[(e, ts)] = []


In [40]:
### Preprocessing version
covering_starts = {(e, t): [] for e in E for t in T}
occ_times = {}

for e in E:
    dur = int(event_to_duration[e])
    for ts in feasible_starts[e]:
        i0 = T_index[ts]
        block = T[i0:i0 + dur]
        occ_times[(e, ts)] = block
        for t in block:
            covering_starts[(e, t)].append(ts)


In [ ]:
# Model
mb = pyo.ConcreteModel()

mb.E = pyo.Set(initialize=E)
mb.T = pyo.Set(initialize=T, ordered=True)

mb.Y_index = pyo.Set(
    dimen=2,
    initialize=[(e, ts) for e in E for ts in feasible_starts[e]]
)

mb.y = pyo.Var(mb.Y_index, domain=pyo.Binary) # = 1 if event e starts at time ts

# Occupancy indicator
mb.w = pyo.Var(mb.E, mb.T, domain=pyo.Binary)


In [ ]:
### Explicit Feasible Starts Constraints 

start_allowed = {}

for e in E:
    dur = int(event_to_duration[e])
    teachers_e = set(as_list(event_to_teachers[e]))

    for ts in T:
        i0 = T_index[ts]

        # enough periods left in horizon
        if i0 + dur - 1 >= len(T):
            start_allowed[(e, ts)] = 0
            continue

        block = T[i0:i0 + dur]
        day0 = time_to_day[ts]

        # event must start and finish on same day
        if not all(time_to_day[t] == day0 for t in block):
            start_allowed[(e, ts)] = 0
            continue

        # teacher unavailability
        violates_unavailability = False
        for info in unavailability.values():
            teacher = info["teacher"]
            if teacher in teachers_e:
                unavail_set = set(info["times"])
                if any(t in unavail_set for t in block):
                    violates_unavailability = True
                    break

        start_allowed[(e, ts)] = 0 if violates_unavailability else 1

mb.start_allowed = pyo.Param(
    mb.Y_index,
    initialize=start_allowed,
    within=pyo.Binary
)

def valid_start_rule(mb, e, ts):
    return mb.y[e, ts] <= mb.start_allowed[e, ts]
mb.ValidStart = pyo.Constraint(mb.Y_index, rule=valid_start_rule)


In [ ]:

# Each event chooses exactly one start
def one_start_rule(mb, e):
    return sum(mb.y[e, ts] for ts in feasible_starts[e]) == 1
mb.OneStart = pyo.Constraint(mb.E, rule=one_start_rule)


# Link chosen start to occupied times
def occupancy_link_rule(mb, e, t):
    starts_that_cover_t = covering_starts[(e, t)]
    if not starts_that_cover_t:
        return mb.w[e, t] == 0
    return mb.w[e, t] == sum(mb.y[e, ts] for ts in starts_that_cover_t)
mb.OccupancyLink = pyo.Constraint(mb.E, mb.T, rule=occupancy_link_rule)


# No teacher clashes
def teacher_clash_rule(mb, teacher, t):
    terms = [mb.w[e, t] for e in mb.E if teacher in as_list(event_to_teachers[e])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mb.TeacherClash = pyo.Constraint(all_teachers, mb.T, rule=teacher_clash_rule)


# No class clashes
def class_clash_rule(mb, cls, t):
    terms = [mb.w[e, t] for e in mb.E if cls in as_list(event_to_classes[e])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mb.ClassClash = pyo.Constraint(all_classes, mb.T, rule=class_clash_rule)



In [ ]:

# Events in same course must be on different days
event_to_course = dict(zip(df_events["event_id"], df_events["course_reference"]))
courses = sorted(set(event_to_course[e] for e in E))
days = sorted(df_times["day_ref"].dropna().unique().tolist())

course_day_terms = defaultdict(list)
for (e, ts) in mb.Y_index:
    c = event_to_course[e]
    d = time_to_day[ts]
    course_day_terms[(c, d)].append((e, ts))

mb.COURSES = pyo.Set(initialize=courses)
mb.DAYS = pyo.Set(initialize=days)

def event_spread_max_rule(mb, c, d):
    terms = course_day_terms.get((c, d), [])
    if not terms:
        return pyo.Constraint.Skip
    return sum(mb.y[e, ts] for (e, ts) in terms) <= 1
mb.EventSpreadMax = pyo.Constraint(mb.COURSES, mb.DAYS, rule=event_spread_max_rule)


In [ ]:
# Objective
mb.obj = pyo.Objective(expr=0, sense=pyo.minimize)

In [ ]:
### Preprocessing Feasible Starts
seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(mb)

    solver = pyo.SolverFactory("gurobi")
    solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    # Manual timing
    start_time = time_module.time()
    res = solver.solve(mb, tee=False)
    elapsed_time = time_module.time() - start_time

    solve_times.append(elapsed_time)
    objectives.append(pyo.value(mb.obj))
    num_vars.append(mb.nvariables())
    num_cons.append(mb.nconstraints())

    solve_status.append(res.solver.termination_condition)

    # Store solution
    seed_solution = {}
    for (e, ts) in mb.Y_index:
        val = pyo.value(mb.y[e, ts], exception=False)
        seed_solution[(e, ts)] = 0.0 if val is None else float(val)
    start_x_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x = start_x_list[0]
res = res

# Calculate averages
avg_time = sum(solve_times) / len(solve_times)
avg_obj = sum(objectives) / len(objectives)
avg_vars = sum(num_vars) / len(num_vars)
avg_cons = sum(num_cons) / len(num_cons)

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {avg_obj:.6f}")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Termination Status: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    scheduled = sum(1 for v in seed_sol.values() if v > 0)
    print(f"  Seed {seeds[i]}: {scheduled} events scheduled in {solve_times[i]:.2f}s, Obj={objectives[i]:.6f}")


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 12.9602
Average Objective: 0.000000
Average Variables: 6030
Average Constraints: 4864
Termination Status: {<TerminationCondition.optimal: 'optimal'>}

Phase 1 produced 10 solutions (one per seed)
  Seed 10: 169 events scheduled in 12.92s, Obj=0.000000
  Seed 20: 169 events scheduled in 8.12s, Obj=0.000000
  Seed 30: 169 events scheduled in 18.26s, Obj=0.000000
  Seed 40: 169 events scheduled in 7.07s, Obj=0.000000
  Seed 50: 169 events scheduled in 14.74s, Obj=0.000000
  Seed 60: 169 events scheduled in 20.51s, Obj=0.000000
  Seed 70: 169 events scheduled in 11.99s, Obj=0.000000
  Seed 80: 169 events scheduled in 19.73s, Obj=0.000000
  Seed 90: 169 events scheduled in 8.22s, Obj=0.000000
  Seed 100: 169 events scheduled in 8.05s, Obj=0.000000


In [ ]:
### Explicit Feasible Starts Constraints
seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_x_list = []

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(mb)

    solver = pyo.SolverFactory("gurobi")
    solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    # Manual timing
    start_time = time_module.time()
    res = solver.solve(mb, tee=False)
    elapsed_time = time_module.time() - start_time

    solve_times.append(elapsed_time)
    objectives.append(pyo.value(mb.obj))
    num_vars.append(mb.nvariables())
    num_cons.append(mb.nconstraints())

    solve_status.append(res.solver.termination_condition)

    # Store solution
    seed_solution = {}
    for (e, ts) in mb.Y_index:
        val = pyo.value(mb.y[e, ts], exception=False)
        seed_solution[(e, ts)] = 0.0 if val is None else float(val)
    start_x_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x = start_x_list[0]
res = res

# Calculate averages
avg_time = sum(solve_times) / len(solve_times)
avg_obj = sum(objectives) / len(objectives)
avg_vars = sum(num_vars) / len(num_vars)
avg_cons = sum(num_cons) / len(num_cons)

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {avg_obj:.6f}")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Termination Status: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_x_list):
    scheduled = sum(1 for v in seed_sol.values() if v > 0)
    print(f"  Seed {seeds[i]}: {scheduled} events scheduled in {solve_times[i]:.2f}s, Obj={objectives[i]:.6f}")


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 11.0146
Average Objective: 0.000000
Average Variables: 6760
Average Constraints: 8379
Termination Status: {<TerminationCondition.optimal: 'optimal'>}

Phase 1 produced 10 solutions (one per seed)
  Seed 10: 169 events scheduled in 7.16s, Obj=0.000000
  Seed 20: 169 events scheduled in 7.74s, Obj=0.000000
  Seed 30: 169 events scheduled in 5.59s, Obj=0.000000
  Seed 40: 169 events scheduled in 17.43s, Obj=0.000000
  Seed 50: 169 events scheduled in 12.79s, Obj=0.000000
  Seed 60: 169 events scheduled in 6.66s, Obj=0.000000
  Seed 70: 169 events scheduled in 6.54s, Obj=0.000000
  Seed 80: 169 events scheduled in 12.65s, Obj=0.000000
  Seed 90: 169 events scheduled in 12.52s, Obj=0.000000
  Seed 100: 169 events scheduled in 21.07s, Obj=0.000000


### Pure Occupancy

In [ ]:
# Build per-day ordered time lists and previous/next time-slot lookups.
times_by_day = {}
for _, row in df_times.sort_values("order_index").iterrows():
    d = row["day_ref"]
    times_by_day.setdefault(d, []).append(row["time_id"])

days = list(times_by_day.keys())

prev_in_day = {}
next_in_day = {}

for d in days:
    day_times = times_by_day[d]
    for i, t in enumerate(day_times):
        prev_in_day[t] = None if i == 0 else day_times[i - 1]
        next_in_day[t] = None if i == len(day_times) - 1 else day_times[i + 1]




# Build illegal starts for full occupancy model (timeslots at which an event is forbidden from starting)
illegal_starts = []

for e in E:
    dur = int(event_to_duration[e])
    teachers_e = set(as_list(event_to_teachers[e]))

    for ts in T:
        i0 = T_index[ts]

        # if not enough periods remain, skip because no full block exists
        if i0 + dur - 1 >= len(T):
            continue

        block = T[i0:i0 + dur]
        day0 = time_to_day[ts]

        # must start and finish on the same day
        same_day = all(time_to_day[t] == day0 for t in block)

        # no assigned teacher may be unavailable in any occupied period
        violates_unavailability = False
        for info in unavailability.values():
            teacher = info["teacher"]
            if teacher in teachers_e:
                unavail_set = set(info["times"])
                if any(t in unavail_set for t in block):
                    violates_unavailability = True
                    break

        # if this start would be illegal, forbid the full block
        if (not same_day) or violates_unavailability:
            illegal_starts.append((e, ts))

In [ ]:
mc = pyo.ConcreteModel()

mc.E = pyo.Set(initialize=E)
mc.T = pyo.Set(initialize=T, ordered=True)
mc.D = pyo.Set(initialize=days)

mc.w = pyo.Var(mc.E, mc.T, domain=pyo.Binary)


# Define binary variables over each event-day-time combination.
# u[e,d,t] and v[e,d,t] are 1 if event e occupies timeslot t on day d.
mc.U_index = pyo.Set(
    dimen=3,
    initialize=[(e, d, t) for e in E for d in days for t in times_by_day[d]]
)
mc.u = pyo.Var(mc.U_index, domain=pyo.Binary)
mc.v = pyo.Var(mc.U_index, domain=pyo.Binary)




In [ ]:
# Enforce each event to occupy exactly its required number of timeslots.
def duration_rule(mc, e):
    return sum(mc.w[e, t] for t in mc.T) == int(event_to_duration[e])
mc.Duration = pyo.Constraint(mc.E, rule=duration_rule)

#  Forbid illegal event placements 
mc.ILLEGAL_STARTS = pyo.Set(dimen=2, initialize=illegal_starts)

def illegal_block_rule(mc, e, ts):
    dur = int(event_to_duration[e])
    i0 = T_index[ts]
    block = T[i0:i0 + dur]
    return sum(mc.w[e, t] for t in block) <= dur - 1

mc.IllegalBlock = pyo.Constraint(mc.ILLEGAL_STARTS, rule=illegal_block_rule)


# No teacher clashes
def teacher_clash_rule(mc, teacher, t):
    terms = [mc.w[e, t] for e in mc.E if teacher in as_list(event_to_teachers[e])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mc.TeacherClash = pyo.Constraint(all_teachers, mc.T, rule=teacher_clash_rule)


# No class clashes
def class_clash_rule(mc, cls, t):
    terms = [mc.w[e, t] for e in mc.E if cls in as_list(event_to_classes[e])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mc.ClassClash = pyo.Constraint(all_classes, mc.T, rule=class_clash_rule)

In [ ]:
# Mark the start of an event block within a day: u[e,d,t] becomes 1 when
# event e is scheduled at t and not in the previous timeslot of that day.
def start_edge_rule(mc, e, d, t):
    p = prev_in_day[t]
    if p is None:
        return mc.u[e, d, t] >= mc.w[e, t]
    return mc.u[e, d, t] >= mc.w[e, t] - mc.w[e, p]
mc.StartEdge = pyo.Constraint(mc.U_index, rule=start_edge_rule)

# Mark the end of an event block within a day: v[e,d,t] becomes 1 when
# event e is scheduled at t and not in the next timeslot of that day.
def end_edge_rule(mc, e, d, t):
    n = next_in_day[t]
    if n is None:
        return mc.v[e, d, t] >= mc.w[e, t]
    return mc.v[e, d, t] >= mc.w[e, t] - mc.w[e, n]
mc.EndEdge = pyo.Constraint(mc.U_index, rule=end_edge_rule)

# Allow at most one start and one end per day for each event.
def one_start_per_day_rule(mc, e, d):
    return sum(mc.u[e, d, t] for t in times_by_day[d]) <= 1
mc.OneStartPerDay = pyo.Constraint(mc.E, mc.D, rule=one_start_per_day_rule)

def one_end_per_day_rule(mc, e, d):
    return sum(mc.v[e, d, t] for t in times_by_day[d]) <= 1
mc.OneEndPerDay = pyo.Constraint(mc.E, mc.D, rule=one_end_per_day_rule)

# Force each event to have exactly one start and one end over the whole timetable.
def one_start_overall_rule(mc, e):
    return sum(mc.u[e, d, t] for d in mc.D for t in times_by_day[d]) == 1
mc.OneStartOverall = pyo.Constraint(mc.E, rule=one_start_overall_rule)

def one_end_overall_rule(mc, e):
    return sum(mc.v[e, d, t] for d in mc.D for t in times_by_day[d]) == 1
mc.OneEndOverall = pyo.Constraint(mc.E, rule=one_end_overall_rule)



In [ ]:
### Events that are part of the same course must be on different days

mc.day_used = pyo.Var(mc.E, mc.D, domain=pyo.Binary)

def day_used_upper_rule(mc, e, d):
    return sum(mc.w[e, t] for t in times_by_day[d]) <= len(times_by_day[d]) * mc.day_used[e, d]
mc.DayUsedUpper = pyo.Constraint(mc.E, mc.D, rule=day_used_upper_rule)

def day_used_lower_rule(mc, e, d):
    return sum(mc.w[e, t] for t in times_by_day[d]) >= mc.day_used[e, d]
mc.DayUsedLower = pyo.Constraint(mc.E, mc.D, rule=day_used_lower_rule)

event_to_course = dict(zip(df_events["event_id"], df_events["course_reference"]))
courses = sorted(set(event_to_course[e] for e in E))

mc.COURSES = pyo.Set(initialize=courses)

def event_spread_max_rule(mc, c, d):
    terms = [mc.day_used[e, d] for e in mc.E if event_to_course[e] == c]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1
mc.EventSpreadMax = pyo.Constraint(mc.COURSES, mc.D, rule=event_spread_max_rule)


In [ ]:
mc.obj = pyo.Objective(expr=0, sense=pyo.minimize)

In [53]:
### Full Occupancy
seeds = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []
start_w_list = []

import time as time_module

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(mc)

    solver = pyo.SolverFactory("gurobi")
    solver.options["TimeLimit"] = 600
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    # Manual timing
    start_time = time_module.time()
    res = solver.solve(mc, tee=False)
    elapsed_time = time_module.time() - start_time

    solve_times.append(elapsed_time)
    objectives.append(pyo.value(mc.obj))
    num_vars.append(mc.nvariables())
    num_cons.append(mc.nconstraints())
    solve_status.append(res.solver.termination_condition)

    # Store occupancy solution
    seed_solution = {}
    for e in mc.E:
        for t in mc.T:
            val = pyo.value(mc.w[e, t], exception=False)
            seed_solution[(e, t)] = 0.0 if val is None else float(val)
    start_w_list.append(seed_solution)

# Keep compatibility with later cells that expect start_x / res
start_x_list = start_w_list
start_x = start_w_list[0]
res = res

# Calculate averages
avg_time = sum(solve_times) / len(solve_times)
avg_obj = sum(objectives) / len(objectives)
avg_vars = sum(num_vars) / len(num_vars)
avg_cons = sum(num_cons) / len(num_cons)

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {avg_obj:.6f}")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Termination Status: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_w_list)} solutions (one per seed)")
for i, seed_sol in enumerate(start_w_list):
    scheduled = sum(1 for v in seed_sol.values() if v > 0.5)
    print(f"  Seed {seeds[i]}: {scheduled} occupied event-times in {solve_times[i]:.2f}s, Obj={objectives[i]:.6f}")


SOLVER METRICS SUMMARY - PHASE 1
Average Solve Time (s): 9.4476
Average Objective: 0.000000
Average Variables: 10985
Average Constraints: 12796
Termination Status: {<TerminationCondition.optimal: 'optimal'>}

Phase 1 produced 10 solutions (one per seed)
  Seed 10: 200 occupied event-times in 12.66s, Obj=0.000000
  Seed 20: 200 occupied event-times in 11.76s, Obj=0.000000
  Seed 30: 200 occupied event-times in 8.48s, Obj=0.000000
  Seed 40: 200 occupied event-times in 6.23s, Obj=0.000000
  Seed 50: 200 occupied event-times in 5.44s, Obj=0.000000
  Seed 60: 200 occupied event-times in 8.86s, Obj=0.000000
  Seed 70: 200 occupied event-times in 14.04s, Obj=0.000000
  Seed 80: 200 occupied event-times in 7.27s, Obj=0.000000
  Seed 90: 200 occupied event-times in 8.27s, Obj=0.000000
  Seed 100: 200 occupied event-times in 11.46s, Obj=0.000000
